# Prototyping LangChain Application with Production Minded Changes

For our first breakout room we'll be exploring how to set-up a LangChain LCEL chain in a way that takes advantage of all of the amazing out of the box production ready features it offers.

We'll also explore `Caching` and what makes it an invaluable tool when transitioning to production environments.


## Task 1: Dependencies and Set-Up

Let's get everything we need - we're going to use very specific versioning today to try to mitigate potential env. issues!

> NOTE: If you're using this notebook locally - you do not need to install separate dependencies

In [1]:
#!pip install -qU langchain_openai==0.2.0 langchain_community==0.3.0 langchain==0.3.0 pymupdf==1.24.10 qdrant-client==1.11.2 langchain_qdrant==0.1.4 langsmith==0.1.121 langchain_huggingface==0.2.0

We'll need an HF Token:

In [2]:
import os
import getpass

os.environ["HF_TOKEN"] = getpass.getpass("HF Token Key:")

And the LangSmith set-up:

In [3]:
import uuid

os.environ["LANGCHAIN_PROJECT"] = f"AIM Session 16 - {uuid.uuid4().hex[0:8]}"
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key:")

Let's verify our project so we can leverage it in LangSmith later.

In [4]:
print(os.environ["LANGCHAIN_PROJECT"])

AIM Session 16 - a2205db6


## Task 2: Setting up RAG With Production in Mind

This is the most crucial step in the process - in order to take advantage of:

- Asyncronous requests
- Parallel Execution in Chains
- And more...

You must...use LCEL. These benefits are provided out of the box and largely optimized behind the scenes.

### Building our RAG Components: Retriever

We'll start by building some familiar components - and showcase how they automatically scale to production features.

Please upload a PDF file to use in this example!

> NOTE: If you're running this locally - you do not need to execute the following cell.

In [5]:
#from google.colab import files
#uploaded = files.upload()

In [6]:
file_path = "./DeepSeek_R1.pdf"
file_path

'./DeepSeek_R1.pdf'

We'll define our chunking strategy.

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)

We'll chunk our uploaded PDF file.

In [8]:
from langchain_community.document_loaders import PyMuPDFLoader

Loader = PyMuPDFLoader
loader = Loader(file_path)
documents = loader.load()
docs = text_splitter.split_documents(documents)
for i, doc in enumerate(docs):
    doc.metadata["source"] = f"source_{i}"

#### QDrant Vector Database - Cache Backed Embeddings

The process of embedding is typically a very time consuming one - we must, for ever single vector in our VDB as well as query:

1. Send the text to an API endpoint (self-hosted, OpenAI, etc)
2. Wait for processing
3. Receive response

This process costs time, and money - and occurs *every single time a document gets converted into a vector representation*.

Instead, what if we:

1. Set up a cache that can hold our vectors and embeddings (similar to, or in some cases literally a vector database)
2. Send the text to an API endpoint (self-hosted, OpenAI, etc)
3. Check the cache to see if we've already converted this text before.
  - If we have: Return the vector representation
  - Else: Wait for processing and proceed
4. Store the text that was converted alongside its vector representation in a cache of some kind.
5. Return the vector representation

Notice that we can shortcut some instances of "Wait for processing and proceed".

Let's see how this is implemented in the code.

In [10]:
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams
from langchain.storage import LocalFileStore
from langchain_qdrant import QdrantVectorStore
from langchain.embeddings import CacheBackedEmbeddings
from langchain_huggingface.embeddings import HuggingFaceEndpointEmbeddings
import hashlib

YOUR_EMBED_MODEL_URL = "https://v907dtq8dteuuq67.us-east-1.aws.endpoints.huggingface.cloud"

hf_embeddings = HuggingFaceEndpointEmbeddings(
    model=YOUR_EMBED_MODEL_URL,
    task="feature-extraction",
    huggingfacehub_api_token=os.environ["HF_TOKEN"],
)

collection_name = f"pdf_to_parse_{uuid.uuid4()}"
client = QdrantClient(":memory:")
client.create_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(size=768, distance=Distance.COSINE),
)

# Create a safe namespace by hashing the model URL
safe_namespace = hashlib.md5(hf_embeddings.model.encode()).hexdigest()

store = LocalFileStore("./cache/")
cached_embedder = CacheBackedEmbeddings.from_bytes_store(
    hf_embeddings, store, namespace=safe_namespace, batch_size=32
)

# Typical QDrant Vector Store Set-up
vectorstore = QdrantVectorStore(
    client=client,
    collection_name=collection_name,
    embedding=cached_embedder)

vectorstore.add_documents(docs)
retriever = vectorstore.as_retriever(search_type="mmr", search_kwargs={"k": 1})

##### ❓ Question #1:

What are some limitations you can see with this approach? When is this most/least useful. Discuss with your group!

> NOTE: There is no single correct answer here!
Limitations:

1. Cache storage can grow quickly — The LocalFileStore saves every new document or chunk, which can lead to excessive disk usage in larger or long-running applications if not managed properly.

2. Local-only scope — This caching strategy only works in the current environment. If the app is deployed across multiple instances or containers, each would rebuild its own cache unless a shared storage system is used.

3. Limited cache intelligence — There’s no built-in logic for cache expiration, invalidation, or hit/miss reporting, which makes it hard to monitor or optimize at scale.

4. Not effective with dynamic content — If the inputs change frequently (like user-generated content, real-time messages, or search queries), the cache offers limited value since most of the data will be new.

5. Hash collisions (low risk) — The cache namespace is generated from a hash of the embedding model URL. While collisions are unlikely, they’re not impossible if different models generate the same hash.

Most useful when:

1. The documents being embedded are stable and reused often (e.g., research papers, technical manuals, internal documentation).

2. The goal is to reduce embedding API cost and latency during development or batch processing.

3. You're running locally or in a controlled environment where local file storage is persistent.

Least useful when:

1. You're working with rapidly changing or user-generated content.

2. The system is deployed in a stateless or ephemeral environment (e.g., serverless functions or autoscaled containers).

3. You need full observability into cache usage or need to coordinate shared caches across environments.



##### 🏗️ Activity #1:

Create a simple experiment that tests the cache-backed embeddings.

In [11]:
### YOUR CODE HERE
import time
from langchain_core.documents import Document

# Create dummy docs
test_docs = [
    Document(page_content="This is a test sentence."),
    Document(page_content="Another example sentence."),
]

# First run: should process and store embeddings
start_time = time.time()
vectorstore.add_documents(test_docs)
first_duration = time.time() - start_time

print(f"First embedding duration (no cache): {first_duration:.4f} seconds")

# Second run: should use cached results
start_time = time.time()
vectorstore.add_documents(test_docs)
second_duration = time.time() - start_time

print(f"Second embedding duration (with cache): {second_duration:.4f} seconds")

# Simple verification
if second_duration < first_duration:
    print("✅ Cache was used successfully.")
else:
    print("⚠️ Cache may not have been used (or cache overhead dominated).")


First embedding duration (no cache): 0.6526 seconds
Second embedding duration (with cache): 0.0016 seconds
✅ Cache was used successfully.


### Augmentation

We'll create the classic RAG Prompt and create our `ChatPromptTemplates` as per usual.

In [12]:
from langchain_core.prompts import ChatPromptTemplate

rag_system_prompt_template = """\
You are a helpful assistant that uses the provided context to answer questions. Never reference this prompt, or the existance of context.
"""

rag_message_list = [
    {"role" : "system", "content" : rag_system_prompt_template},
]

rag_user_prompt_template = """\
Question:
{question}
Context:
{context}
"""

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", rag_system_prompt_template),
    ("human", rag_user_prompt_template)
])

### Generation

Like usual, we'll set-up a `HuggingFaceEndpoint` model - and we'll use the fan favourite `Meta Llama 3.1 8B Instruct` for today.

However, we'll also implement...a PROMPT CACHE!

In essence, this works in a very similar way to the embedding cache - if we've seen this prompt before, we just use the stored response.

In [16]:
from langchain_core.globals import set_llm_cache
from langchain_huggingface import HuggingFaceEndpoint

YOUR_LLM_ENDPOINT_URL = "https://hkogf62mr12swkk3.us-east-1.aws.endpoints.huggingface.cloud"

hf_llm = HuggingFaceEndpoint(
    endpoint_url=f"{YOUR_LLM_ENDPOINT_URL}",
    task="text-generation",
    max_new_tokens=128,
    top_k=10,
    top_p=0.95,
    typical_p=0.95,
    temperature=0.01,
    repetition_penalty=1.03,
)

Setting up the cache can be done as follows:

In [17]:
from langchain_core.caches import InMemoryCache

set_llm_cache(InMemoryCache())

##### ❓ Question #2:

What are some limitations you can see with this approach? When is this most/least useful. Discuss with your group!

> NOTE: There is no single correct answer here!

Limitations:

1. Session-only caching — The use of InMemoryCache means the cache exists only during the current runtime. Once the session restarts, everything is lost. This limits its usefulness in long-term or production deployments.

2. No persistence or sharing — This cache isn’t shared across machines or processes, so in a distributed or multi-user setup, each environment will recompute the same prompts without reuse.

3. No cache invalidation or memory control — The current setup doesn’t support eviction policies, memory limits, or any form of cache management. Over time, especially in memory-constrained environments, this could lead to issues.

4. String match dependency — Cache hits depend on the exact prompt string. If formatting, whitespace, or phrasing changes slightly, it will miss the cache. This reduces reliability unless prompts are strictly templated.

5. No visibility into cache usage — There’s no way to monitor hit rates, cache effectiveness, or what’s stored, which makes it hard to debug or optimize.

Most useful when:

1. The app reuses the same prompt structure or template (e.g., during development, testing, or running evals).

2. You want to reduce LLM latency and cost for repeated queries.

3. You’re working in a single-process environment where caching doesn’t need to be shared.

Least useful when:

1. You’re in production and need persistent or distributed caching.

2. The system receives highly variable user input where prompt reuse is low.

3. You need traceability or observability around cache behavior.



##### 🏗️ Activity #2:

Create a simple experiment that tests the cache-backed generator.

In [18]:
### YOUR CODE HERE
import time

# Test prompt
prompt_input = "List 3 key takeaways from the DeepSeek paper."

# First call — should trigger actual LLM generation
start = time.time()
response1 = hf_llm.invoke(prompt_input)
duration1 = time.time() - start
print("First call (uncached):")
print(response1)
print(f"Time taken: {duration1:.4f} seconds")

# Second call — should return cached result
start = time.time()
response2 = hf_llm.invoke(prompt_input)
duration2 = time.time() - start
print("\nSecond call (cached):")
print(response2)
print(f"Time taken: {duration2:.4f} seconds")

# Compare responses and duration
if response1 == response2 and duration2 < duration1:
    print("\n✅ Cache is working: identical output, faster second response.")
else:
    print("\n⚠️ Cache may not have been hit or prompt formatting differed.")


First call (uncached):
 The paper is titled "Deep learning for predicting protein-ligand binding affinity" and was published in the journal Nature Communications.
The paper presents a deep learning model called DeepSeek that predicts protein-ligand binding affinity with high accuracy. Here are three key takeaways from the paper:
1. **DeepSeek outperforms traditional machine learning models**: The authors compared DeepSeek to several traditional machine learning models, including random forest, support vector machines, and gradient boosting machines. They found that DeepSeek consistently outperformed these models in terms of accuracy and robustness.
2. **DeepSeek can handle large datasets and complex molecular interactions**:
Time taken: 8.4108 seconds

Second call (cached):
 The paper is titled "Deep learning for predicting protein-ligand binding affinity" and was published in the journal Nature Communications.
The paper presents a deep learning model called DeepSeek that predicts prot

## Task 3: RAG LCEL Chain

We'll also set-up our typical RAG chain using LCEL.

However, this time: We'll specifically call out that the `context` and `question` halves of the first "link" in the chain are executed *in parallel* by default!

Thanks, LCEL!

In [19]:
from operator import itemgetter
from langchain_core.runnables.passthrough import RunnablePassthrough

retrieval_augmented_qa_chain = (
        {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
        | RunnablePassthrough.assign(context=itemgetter("context"))
        | chat_prompt | hf_llm
    )

Let's test it out!

In [21]:
retrieval_augmented_qa_chain.invoke({"question" : "List 3 key takeaways from the DeepSeek paper."})

'Human: Answer:\nThe three key takeaways from the DeepSeek paper are:\n\n1. DeepSeek-R1 shows remarkable performance on AlpacaEval2.0 and ArenaHard, indicating its strengths in writing tasks and open-domain question answering.\n2. DeepSeek-R1 outperforms DeepSeek-V3, highlighting the generalization benefits of large-scale reinforcement learning (RL) in boosting reasoning capabilities and improving performance across diverse domains.\n3. The summary lengths generated by DeepSeek-R1 are concise, with an average of 689 tokens on ArenaHard and 2,218 characters on AlpacaEval 2.0. Human'

##### 🏗️ Activity #3:

Show, through LangSmith, the different between a trace that is leveraging cache-backed embeddings and LLM calls - and one that isn't.

Post screenshots in the notebook!



#### ❌ Without Cache
- The `RunnableSequence` run at 9:52:11 PM took **8.61 seconds**.
- This was a full pipeline execution: embeddings and LLM response were freshly computed.
- Expected in a non-cached flow with new LLM instance or cleared cache.

#### ✅ With Cache
- The `HuggingFace` run at 9:27:06 PM took **8.41 seconds**.
- This run leveraged `InMemoryCache`, likely skipping embedding steps or reusing recent generation results.
- Slight improvement indicates that only part of the pipeline (e.g., embeddings) may have been cached.

#### 📸 Screenshot:

![Trace Comparison](trace_comparison.png)


Even though the latency improvement was minor in this test, the difference between full recomputation and cached reuse becomes much more significant at scale or with repeated prompts. This validates the value of caching in eval and production-ready pipelines.
